# NB5 — Modèles neuronaux séquentiels

Notebook des pipelines **P21 à P23**.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"
MODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"

%cd "{MODELS_DIR}"

import sys
if MODELS_DIR not in sys.path:
    sys.path.append(MODELS_DIR)

print("Projet :", PROJECT_ROOT)
print("Dossier courant :", MODELS_DIR)

Mounted at /content/drive
/content/drive/MyDrive/Disaster-Tweets-NLP/notebooks/models_training
Projet : /content/drive/MyDrive/Disaster-Tweets-NLP
Dossier courant : /content/drive/MyDrive/Disaster-Tweets-NLP/notebooks/models_training


## Portée du notebook

Ce notebook compare les architectures neuronales séquentielles classiques sur texte tokenisé :
- `CNN1D` pour les motifs locaux ;
- `BiLSTM` pour les dépendances séquentielles ;
- `BiGRU` comme alternative plus légère.

Ici, **le split train / test existe déjà** dans `../../data/`.
On crée seulement un **jeu de validation stratifié à partir du train**.

In [4]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn tensorflow openpyxl

from collections import OrderedDict

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    stratified_validation_split,
    evaluate_probability_outputs,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
)

seed_everything(42)
print("TensorFlow :", tf.__version__)

TensorFlow : 2.19.0


In [5]:
DATA_DIR = "../../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False

VAL_SIZE_WITHIN_TRAIN = 0.10
RANDOM_STATE = 42

MAX_VOCAB = 20000
MAX_LEN = 50
EMBED_DIM = 128
BATCH_SIZE = 64
EPOCHS = 8
EARLY_STOPPING_PATIENCE = 2

OUTPUT_STEM = "NB5_sequence_neural_models"
RESULTS_DIR = "results"

In [6]:
df_train, X_train_full, y_train_full, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

X_train, X_val, y_train, y_val = stratified_validation_split(
    X_train_full,
    y_train_full,
    val_size=VAL_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
)

print("Taille train :", len(X_train))
print("Taille validation :", len(X_val))
print("Taille test :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - validation :")
print(y_val.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())

Taille train : 8186
Taille validation : 910
Taille test : 2274

Distribution des classes - train :
target
0    0.814073
1    0.185927
Name: proportion, dtype: float64

Distribution des classes - validation :
target
0    0.814286
1    0.185714
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


In [7]:
vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN,
    standardize="lower_and_strip_punctuation",
)

vectorizer.adapt(np.array(X_train))

def build_tf_dataset(texts, labels, batch_size=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((np.array(texts), np.array(labels)))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts), seed=42)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_tf_dataset(X_train, y_train, shuffle=True)
val_ds = build_tf_dataset(X_val, y_val, shuffle=False)
test_ds = build_tf_dataset(X_test, y_test, shuffle=False)

In [8]:
def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="roc_auc"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )
    return model


def build_cnn1d():
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = layers.Embedding(MAX_VOCAB, EMBED_DIM)(x)
    x = layers.Conv1D(128, 3, activation="relu")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="cnn1d")
    return compile_model(model)


def build_bilstm():
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = layers.Embedding(MAX_VOCAB, EMBED_DIM)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="bilstm")
    return compile_model(model)


def build_bigru():
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = layers.Embedding(MAX_VOCAB, EMBED_DIM)(x)
    x = layers.Bidirectional(layers.GRU(64))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="bigru")
    return compile_model(model)


models = OrderedDict({
    "P21_CNN1D": build_cnn1d,
    "P22_BiLSTM": build_bilstm,
    "P23_BiGRU": build_bigru,
})

In [9]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    )
]

results = []
histories = {}

for name, builder in models.items():
    print(f"\nEntraînement -> {name}")
    model = builder()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[name] = history.history

    train_scores = model.predict(build_tf_dataset(X_train, y_train, shuffle=False), verbose=0).ravel()
    test_scores = model.predict(test_ds, verbose=0).ravel()
    result = evaluate_probability_outputs(
        name=name,
        y_train=np.array(y_train),
        train_scores=train_scores,
        y_test=np.array(y_test),
        test_scores=test_scores,
        threshold=0.5,
    )
    results.append(result)

results_df = round_results(pd.DataFrame(results))
results_df


Entraînement -> P21_CNN1D
Epoch 1/8
128/128 ━━━━━━━━━━━━━━━━━━━━ 10s 64ms/step - accuracy: 0.8191 - loss: 0.4463 - pr_auc: 0.3416 - precision: 0.7470 - recall: 0.0407 - roc_auc: 0.6908 - val_accuracy: 0.8593 - val_loss: 0.3417 - val_pr_auc: 0.6861 - val_precision: 0.8868 - val_recall: 0.2781 - val_roc_auc: 0.8619
Epoch 2/8
128/128 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.9109 - loss: 0.2310 - pr_auc: 0.8287 - precision: 0.8340 - recall: 0.6505 - roc_auc: 0.9367 - val_accuracy: 0.8934 - val_loss: 0.2836 - val_pr_auc: 0.7705 - val_precision: 0.8000 - val_recall: 0.5680 - val_roc_auc: 0.9025
Epoch 3/8
128/128 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - accuracy: 0.9742 - loss: 0.0806 - pr_auc: 0.9734 - precision: 0.9518 - recall: 0.9074 - roc_auc: 0.9918 - val_accuracy: 0.9011 - val_loss: 0.3351 - val_pr_auc: 0.7652 - val_precision: 0.7926 - val_recall: 0.6331 - val_roc_auc: 0.9008
Epoch 4/8
128/128 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.9943 - loss: 0.0221 - pr_auc: 0.9974 - pre

pipeline,P21_CNN1D,P22_BiLSTM,P23_BiGRU
train_accuracy,0.9794,0.9395,0.9320
train_precision_macro,0.9775,0.9019,0.9367
train_recall_macro,0.9536,0.8972,0.8327
train_f1_macro,0.9650,0.8995,0.8731
train_precision_weighted,0.9793,0.9392,0.9326
train_recall_weighted,0.9794,0.9395,0.9320
train_f1_weighted,0.9791,0.9394,0.9274
train_precision_class_0,0.9803,0.9613,0.9303
train_recall_class_0,0.9946,0.9646,0.9907
train_f1_class_0,0.9874,0.9629,0.9595


In [10]:
display(results_df)
metric_view = round_results(metric_matrix_from_results(results_df))
display(metric_view)

save_results_bundle(results_df, output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans ./{RESULTS_DIR}")

pipeline,P21_CNN1D,P22_BiLSTM,P23_BiGRU
train_accuracy,0.9794,0.9395,0.9320
train_precision_macro,0.9775,0.9019,0.9367
train_recall_macro,0.9536,0.8972,0.8327
train_f1_macro,0.9650,0.8995,0.8731
train_precision_weighted,0.9793,0.9392,0.9326
train_recall_weighted,0.9794,0.9395,0.9320
train_f1_weighted,0.9791,0.9394,0.9274
train_precision_class_0,0.9803,0.9613,0.9303
train_recall_class_0,0.9946,0.9646,0.9907
train_f1_class_0,0.9874,0.9629,0.9595


pipeline,P21_CNN1D,P22_BiLSTM,P23_BiGRU
train_accuracy,0.9794,0.9395,0.9320
train_precision_macro,0.9775,0.9019,0.9367
train_recall_macro,0.9536,0.8972,0.8327
train_f1_macro,0.9650,0.8995,0.8731
train_precision_weighted,0.9793,0.9392,0.9326
train_recall_weighted,0.9794,0.9395,0.9320
train_f1_weighted,0.9791,0.9394,0.9274
train_precision_class_0,0.9803,0.9613,0.9303
train_recall_class_0,0.9946,0.9646,0.9907
train_f1_class_0,0.9874,0.9629,0.9595


Fichiers CSV/XLSX enregistrés dans ./results
